In [1]:


import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import NETWORK_TYPE, RF_PARAM_5G

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt('data/random_seeds.csv', dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ['toa_pps', 'toa_cir', 'toa_cov', 'campaign_id']
df['measurements_matrix'] = df['measurements_matrix'].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

selected_campaigns = list(range(1, 20))
# Data filtering
df = filter_dataframe(
    df=df,
    include_columns=['pci', 'beam_index', 'nr_arfcn', 'operator_id', 'rsrq', 'sinr', 'rssi', 'rsrp'],
    campaigns=selected_campaigns,
)

Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


In [11]:

from scripts.matrix_operations import create_point_matrix
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scripts.utils import extract_unique_npcis
import pandas as pd
from concurrent.futures import ThreadPoolExecutor


def train_kmeans(df: pd.DataFrame, n_clusters: int, random_state: int, rf_param, unique_npcis, feature_mode: str):
    df = df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    if feature_mode == 'position':
        df_features = df[['lat', 'lng']].values
    elif feature_mode == 'radio':
        df_features, _ = create_point_matrix(df, unique_npcis, rf_param)
    else:
        print(f'Invalid feature mode: {feature_mode}')
        return

    kmeans = KMeans(n_clusters=n_clusters, random_state=random_state)
    cluster_labels = kmeans.fit_predict(df_features)

    df['cluster_labels'] = cluster_labels

    # Calculate inertia
    inertia = kmeans.inertia_

    # Calculate silhouette score
    silhouette_avg = silhouette_score(df_features, cluster_labels)

    # Calculate Davies-Bouldin score
    davies_bouldin = davies_bouldin_score(df_features, cluster_labels)

    return inertia, silhouette_avg, davies_bouldin,


print('KMEANS PERFORMANCE')

# Example usage
unique_npcis = extract_unique_npcis(df['measurements_matrix'])
rf_param = RF_PARAM_5G.RSRQ

data = []
feature_modes = ['position', 'radio']
max_clusters = 20
n_runs = 4
for mode in feature_modes:
    for n_clusters in range(2, max_clusters + 1):
        with ThreadPoolExecutor(max_workers=2) as executor:
            futures = [
                executor.submit(
                    train_kmeans,
                    df,
                    n_clusters,
                    42 * i,
                    rf_param,
                    unique_npcis,
                    mode,
                )
                for i in range(n_runs)
            ]
            for f in futures:
                r = f.result()
                data.append((n_clusters, mode) + r)
                print(f'Testing with k={n_clusters} mode {mode}')

# Print the data to verify its structure
print(data)

# Define columns
cols = ['K', 'Feature', 'Inertia', 'Silhouette Score', 'Davies-Bouldin Score']

# Create DataFrame
res_df = pd.DataFrame(data, columns=cols)

# Print the resulting DataFrame
print(res_df.head(20))

KMEANS PERFORMANCE
Testing with k=2 mode position
Testing with k=2 mode position
Testing with k=2 mode position
Testing with k=2 mode position
Testing with k=3 mode position
Testing with k=3 mode position
Testing with k=3 mode position
Testing with k=3 mode position
Testing with k=4 mode position
Testing with k=4 mode position
Testing with k=4 mode position
Testing with k=4 mode position
Testing with k=5 mode position
Testing with k=5 mode position
Testing with k=5 mode position
Testing with k=5 mode position
Testing with k=6 mode position
Testing with k=6 mode position
Testing with k=6 mode position
Testing with k=6 mode position
Testing with k=7 mode position
Testing with k=7 mode position
Testing with k=7 mode position
Testing with k=7 mode position
Testing with k=8 mode position
Testing with k=8 mode position
Testing with k=8 mode position
Testing with k=8 mode position
Testing with k=9 mode position
Testing with k=9 mode position
Testing with k=9 mode position
Testing with k=9 mod

In [18]:
res_df.to_csv('kmeans_res_2.csv')

means = res_df[res_df['Feature'] == 'position'].groupby(['Feature', 'K']).mean().round(4)

means

Inertia  Silhouette Score  Davies-Bouldin Score
Feature  K                                                  
position 2    0.0079            0.6115                0.5637
         3    0.0036            0.6363                0.5308
         4    0.0022            0.6291                0.5782
         5    0.0014            0.6103                0.6247
         6    0.0011            0.5951                0.6217
         7    0.0008            0.5839                0.5982
         8    0.0007            0.5689                0.6142
         9    0.0006            0.5269                0.6447
         10   0.0005            0.5317                0.6415
         11   0.0004            0.5326                0.6527
         12   0.0004            0.5288                0.6552
         13   0.0003            0.5204                0.6743
         14   0.0003            0.5242                0.6632
         15   0.0003            0.5172                0.6760
         16   0.0003            0.5183                0.6707
         17   0.0002            0.5163                0.6856
         18   0.0002            0.5125                0.6810
         19   0.0002            0.5112                0.6881
         20   0.0002            0.5082                0.6936

In [9]:

from scripts.data_writer import save_experiment_result

r = train_kmeans(
    df,
    n_clusters=10,
    random_state=42 * 10,
    rf_param=rf_param,
    unique_npcis=unique_npcis,
    feature_mode='radio'
)

df['cluster_labels'].value_counts()

df['cluster'] = df['cluster_labels']
df['predicted_cluster'] = df['cluster_labels']

config = {}
data = {'results': df}
save_experiment_result('kmeans_experiment', config, data)

Experiment stored in ./data/results/experiments/kmeans_experiment_1


In [19]:
print(means)

             Inertia  Silhouette Score  Davies-Bouldin Score
Feature  K                                                  
position 2    0.0079            0.6115                0.5637
         3    0.0036            0.6363                0.5308
         4    0.0022            0.6291                0.5782
         5    0.0014            0.6103                0.6247
         6    0.0011            0.5951                0.6217
         7    0.0008            0.5839                0.5982
         8    0.0007            0.5689                0.6142
         9    0.0006            0.5269                0.6447
         10   0.0005            0.5317                0.6415
         11   0.0004            0.5326                0.6527
         12   0.0004            0.5288                0.6552
         13   0.0003            0.5204                0.6743
         14   0.0003            0.5242                0.6632
         15   0.0003            0.5172                0.6760
         16   0.0003    

In [21]:
for i, row in means.iterrows():
    items = [str(e) for e in row.values.tolist()]
    r = f"{i[1]} & {' & '.join(items)} \\\\ \hline"
    print(r)

2 & 0.0079 & 0.6115 & 0.5637 \\ \hline
3 & 0.0036 & 0.6363 & 0.5308 \\ \hline
4 & 0.0022 & 0.6291 & 0.5782 \\ \hline
5 & 0.0014 & 0.6103 & 0.6247 \\ \hline
6 & 0.0011 & 0.5951 & 0.6217 \\ \hline
7 & 0.0008 & 0.5839 & 0.5982 \\ \hline
8 & 0.0007 & 0.5689 & 0.6142 \\ \hline
9 & 0.0006 & 0.5269 & 0.6447 \\ \hline
10 & 0.0005 & 0.5317 & 0.6415 \\ \hline
11 & 0.0004 & 0.5326 & 0.6527 \\ \hline
12 & 0.0004 & 0.5288 & 0.6552 \\ \hline
13 & 0.0003 & 0.5204 & 0.6743 \\ \hline
14 & 0.0003 & 0.5242 & 0.6632 \\ \hline
15 & 0.0003 & 0.5172 & 0.676 \\ \hline
16 & 0.0003 & 0.5183 & 0.6707 \\ \hline
17 & 0.0002 & 0.5163 & 0.6856 \\ \hline
18 & 0.0002 & 0.5125 & 0.681 \\ \hline
19 & 0.0002 & 0.5112 & 0.6881 \\ \hline
20 & 0.0002 & 0.5082 & 0.6936 \\ \hline
